In [119]:
%pip install pandas numpy xgboost scikit-learn plotly optuna

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [120]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import plotly.graph_objects as go
import plotly.io as pio
import optuna
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ----------------------------------------------------------
# ⚙️  CONFIGURATION
# ----------------------------------------------------------
CSV_PATH = r"c:\Users\User\Pictures\For XGBoost\2016 - 2025 datasets.csv"

# Set to 'browser' to open charts in browser, or 'png' to save images
PLOTLY_RENDERER = "browser"
pio.renderers.default = PLOTLY_RENDERER

In [121]:
# ============================================================
# 1. LOAD DATA
# ============================================================
print("\n📁 Loading data from:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
print(f"✅ File loaded: {len(df)} rows")
print("\nRaw data (first 5 rows):")
print(df.head())


📁 Loading data from: c:\Users\User\Pictures\For XGBoost\2016 - 2025 datasets.csv
✅ File loaded: 120 rows

Raw data (first 5 rows):
   Year     Month  Arrivals  Peak_Season  Philippine_Holidays  \
0  2016   January     57498            0                    2   
1  2016  February     65097            0                    2   
2  2016     March     58104            0                    3   
3  2016     April     56530            0                    1   
4  2016       May     58776            0                    2   

   Top10Market_Holidays  Avg_HighTemp  Avg_LowTemp  Precipitation  \
0                    17            31           24           0.10   
1                    20            30           24           0.17   
2                     9            32           24           0.04   
3                     6            32           25           0.07   
4                    22            33           26           0.29   

   Inflation_Rate  is_December  is_Lockdown  
0             1.

In [122]:
# ============================================================
# 2. DATA PREPROCESSING
# ============================================================
df['Date'] = pd.to_datetime(df['Year'].astype(str) + ' ' + df['Month'], format='%Y %B')
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"\n✅ Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"✅ Total records: {len(df)} months")
print("\nProcessed data (first 5 rows):")
print(df.head())


✅ Date range: 2016-01-01 to 2025-12-01
✅ Total records: 120 months

Processed data (first 5 rows):
            Year     Month  Arrivals  Peak_Season  Philippine_Holidays  \
Date                                                                     
2016-01-01  2016   January     57498            0                    2   
2016-02-01  2016  February     65097            0                    2   
2016-03-01  2016     March     58104            0                    3   
2016-04-01  2016     April     56530            0                    1   
2016-05-01  2016       May     58776            0                    2   

            Top10Market_Holidays  Avg_HighTemp  Avg_LowTemp  Precipitation  \
Date                                                                         
2016-01-01                    17            31           24           0.10   
2016-02-01                    20            30           24           0.17   
2016-03-01                     9            32           24          

In [123]:
# ============================================================
# 3. EVALUATION HELPER & SHARED MAPS
# ============================================================
def evaluate_model(y_true, y_pred, dataset_name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    mask = y_true != 0
    if mask.any():
        mape     = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
        accuracy = 100 - mape
    else:
        mape, accuracy = 0, 0

    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape, 'accuracy': accuracy}


# Future date range (next 12 months after last data point)
last_date  = df.index.max()
future_12m = pd.date_range(
    start=last_date + pd.DateOffset(months=1), periods=12, freq='MS'
)

# Shared weather lookup maps (used by V2 and V3 future predictions)
temp_high_map  = {1:29,2:29,3:30,4:31,5:32,6:32,7:32,8:32,9:32,10:31,11:30,12:29}
temp_low_map   = {1:25,2:25,3:25,4:26,5:26,6:26,7:26,8:26,9:26,10:26,11:25,12:25}
precip_map     = {1:0.7,2:0.5,3:0.4,4:0.3,5:0.4,6:0.7,7:0.9,8:0.8,9:0.9,10:1.0,11:0.9,12:0.7}
ph_holiday_map = {1:2,2:1,3:2,4:3,5:2,6:2,7:1,8:2,9:1,10:1,11:3,12:4}

# 80/20 split point (computed once, reused across versions)
split_point = int(len(df) * 0.8)

print("✅ Helpers and shared maps ready.")

✅ Helpers and shared maps ready.


In [124]:
# ============================================================
# 4. VERSION 1.0 — Seasonal & Holiday Features Only
# ============================================================
print("\n" + "="*80)
print("🏖️  XGBoost VERSION 1.0")
print("="*80)

df_v1 = df.copy()
df_v1['Year']      = df_v1.index.year
df_v1['Month_Num'] = df_v1.index.month
df_v1 = df_v1.dropna()

features_v1 = ['Peak_Season', 'Philippine_Holidays', 'Top10Market_Holidays']
X_v1 = df_v1[features_v1]
y_v1 = df_v1['Arrivals']

X_train_v1, X_test_v1 = X_v1.iloc[:split_point], X_v1.iloc[split_point:]
y_train_v1, y_test_v1 = y_v1.iloc[:split_point], y_v1.iloc[split_point:]

print(f"\nVersion 1.0 Data Split:")
print(f"  Training: {X_train_v1.index.min().date()} to {X_train_v1.index.max().date()} ({len(X_train_v1)} months)")
print(f"  Testing:  {X_test_v1.index.min().date()} to {X_test_v1.index.max().date()} ({len(X_test_v1)} months)")

model_v1 = xgb.XGBRegressor(
    n_estimators=300, max_depth=2, learning_rate=0.05,
    random_state=42, n_jobs=-1
)
model_v1.fit(X_train_v1, y_train_v1)

y_pred_train_v1_rounded = np.round(model_v1.predict(X_train_v1)).astype(int)
y_pred_test_v1_rounded  = np.round(model_v1.predict(X_test_v1)).astype(int)
train_metrics_v1 = evaluate_model(y_train_v1, y_pred_train_v1_rounded, "V1.0 TRAIN")
test_metrics_v1  = evaluate_model(y_test_v1,  y_pred_test_v1_rounded,  "V1.0 TEST")

print("\nPredicted vs Actual (first 10 test rows):")
print(pd.DataFrame({'Actual': y_test_v1.values, 'Predicted': y_pred_test_v1_rounded}).head(10))

print("\n📈 V1.0 TEST PERFORMANCE:")
print(f"   MAE:      {test_metrics_v1['mae']:,.0f} arrivals")
print(f"   RMSE:     {test_metrics_v1['rmse']:,.0f} arrivals")
print(f"   MAPE:     {test_metrics_v1['mape']:.2f}%")
print(f"   Accuracy: {test_metrics_v1['accuracy']:.2f}%")

# Chart: test predictions
fig_v1_test = go.Figure()
fig_v1_test.add_trace(go.Scatter(x=X_test_v1.index, y=y_test_v1, mode='lines+markers', name='Actual',
    line=dict(color='blue', width=2), marker=dict(size=8)))
fig_v1_test.add_trace(go.Scatter(x=X_test_v1.index, y=y_pred_test_v1_rounded, mode='lines+markers', name='Predicted',
    line=dict(color='red', width=2, dash='dash'), marker=dict(size=8, symbol='circle')))
fig_v1_test.update_layout(title='V1.0 - Actual vs Predicted Arrivals', xaxis_title='Month',
    yaxis_title='Arrivals', hovermode='x unified', template='plotly_white', height=450, width=1000)
fig_v1_test.update_xaxes(tickformat='%b %Y', tickangle=45)

# Future predictions V1
future_v1 = pd.DataFrame(index=future_12m)
future_v1['Peak_Season']          = future_v1.index.month.isin([8, 12]).astype(int)
future_v1['Philippine_Holidays']  = future_v1.index.month.map(ph_holiday_map)
future_v1['Top10Market_Holidays'] = 15

pred_v1_12m_rounded = np.round(model_v1.predict(future_v1[features_v1])).astype(int)

print("\n🔮 V1.0 NEXT 12 MONTHS PREDICTIONS:")
for i, date in enumerate(future_12m):
    print(f"   {date.strftime('%b %Y')}: {pred_v1_12m_rounded[i]:,} arrivals ({pred_v1_12m_rounded[i]/1000:.1f}k)")

fig_v1_12m = go.Figure()
fig_v1_12m.add_trace(go.Scatter(x=future_12m, y=pred_v1_12m_rounded, mode='lines+markers',
    name='V1.0 Predictions', line=dict(color='red', width=3), marker=dict(size=6)))
fig_v1_12m.update_layout(title='V1.0 - Next 12 Months Predictions', xaxis_title='Month',
    yaxis_title='Predicted Arrivals', template='plotly_white', height=400, width=1000)
fig_v1_12m.update_xaxes(tickformat='%b %Y', tickangle=45)


🏖️  XGBoost VERSION 1.0

Version 1.0 Data Split:
  Training: 2016-01-01 to 2023-12-01 (96 months)
  Testing:  2024-01-01 to 2025-12-01 (24 months)

Predicted vs Actual (first 10 test rows):
   Actual  Predicted
0   72264      41992
1   65976      75287
2   67741      60060
3   71858      43627
4   72783      24549
5   66344      53791
6   71180      55996
7   74329      53791
8   63784      46494
9   66896      56037

📈 V1.0 TEST PERFORMANCE:
   MAE:      24,720 arrivals
   RMSE:     28,661 arrivals
   MAPE:     32.58%
   Accuracy: 67.42%

🔮 V1.0 NEXT 12 MONTHS PREDICTIONS:
   Jan 2026: 43,627 arrivals (43.6k)
   Feb 2026: 41,992 arrivals (42.0k)
   Mar 2026: 43,627 arrivals (43.6k)
   Apr 2026: 56,443 arrivals (56.4k)
   May 2026: 43,627 arrivals (43.6k)
   Jun 2026: 43,627 arrivals (43.6k)
   Jul 2026: 41,992 arrivals (42.0k)
   Aug 2026: 24,764 arrivals (24.8k)
   Sep 2026: 41,992 arrivals (42.0k)
   Oct 2026: 41,992 arrivals (42.0k)
   Nov 2026: 56,443 arrivals (56.4k)
   Dec 2026

In [125]:
# ============================================================
# 5. VERSION 2.0 — + Weather & Economic Features
# ============================================================
print("\n" + "="*80)
print("🌦️  XGBoost VERSION 2.0")
print("="*80)

df_v2 = df.copy()
df_v2['Year']      = df_v2.index.year
df_v2['Month_Num'] = df_v2.index.month
df_v2 = df_v2.dropna()

features_v2 = [
    'Peak_Season', 'Philippine_Holidays', 'Top10Market_Holidays',
    'Avg_HighTemp', 'Avg_LowTemp', 'Precipitation',
    'Inflation_Rate', 'is_December', 'is_Lockdown'
]
X_v2 = df_v2[features_v2]
y_v2 = df_v2['Arrivals']

X_train_v2, X_test_v2 = X_v2.iloc[:split_point], X_v2.iloc[split_point:]
y_train_v2, y_test_v2 = y_v2.iloc[:split_point], y_v2.iloc[split_point:]

print(f"\n📊 V2 Data Split:")
print(f"  Training: {X_train_v2.index.min().date()} to {X_train_v2.index.max().date()} ({len(X_train_v2)} months)")
print(f"  Testing:  {X_test_v2.index.min().date()} to {X_test_v2.index.max().date()} ({len(X_test_v2)} months)")

model_v2 = xgb.XGBRegressor(
    n_estimators=300, max_depth=2, learning_rate=0.05,
    reg_alpha=0.5, reg_lambda=1.0, random_state=42, n_jobs=-1
)
model_v2.fit(X_train_v2, y_train_v2)

y_pred_train_v2_rounded = np.round(model_v2.predict(X_train_v2)).astype(int)
y_pred_test_v2_rounded  = np.round(model_v2.predict(X_test_v2)).astype(int)
train_metrics_v2 = evaluate_model(y_train_v2, y_pred_train_v2_rounded, "V2.0 TRAIN")
test_metrics_v2  = evaluate_model(y_test_v2,  y_pred_test_v2_rounded,  "V2.0 TEST")

print("\nPredicted vs Actual (first 10 test rows):")
print(pd.DataFrame({'Actual': y_test_v2.values, 'Predicted': y_pred_test_v2_rounded}).head(10))

print("\n📈 V2.0 TEST PERFORMANCE:")
print(f"   MAE:      {test_metrics_v2['mae']:,.0f} arrivals")
print(f"   RMSE:     {test_metrics_v2['rmse']:,.0f} arrivals")
print(f"   MAPE:     {test_metrics_v2['mape']:.2f}%")
print(f"   Accuracy: {test_metrics_v2['accuracy']:.2f}%")

fig_v2_test = go.Figure()
fig_v2_test.add_trace(go.Scatter(x=X_test_v2.index, y=y_test_v2, mode='lines+markers', name='Actual',
    line=dict(color='blue', width=2), marker=dict(size=8)))
fig_v2_test.add_trace(go.Scatter(x=X_test_v2.index, y=y_pred_test_v2_rounded, mode='lines+markers', name='Predicted',
    line=dict(color='red', width=2, dash='dash'), marker=dict(size=8, symbol='circle')))
fig_v2_test.update_layout(title='V2.0 - Actual vs Predicted Arrivals', xaxis_title='Month',
    yaxis_title='Arrivals', hovermode='x unified', template='plotly_white', height=450, width=1000)
fig_v2_test.update_xaxes(tickformat='%b %Y', tickangle=45)

# Future predictions V2
future_v2 = pd.DataFrame(index=future_12m)
future_v2['Peak_Season']          = future_v2.index.month.isin([8, 12]).astype(int)
future_v2['Philippine_Holidays']  = future_v2.index.month.map(ph_holiday_map)
future_v2['Top10Market_Holidays'] = 15
future_v2['Avg_HighTemp']         = future_v2.index.month.map(temp_high_map)
future_v2['Avg_LowTemp']          = future_v2.index.month.map(temp_low_map)
future_v2['Precipitation']        = future_v2.index.month.map(precip_map)
future_v2['Inflation_Rate']       = 3.0
future_v2['is_December']          = (future_v2.index.month == 12).astype(int)
future_v2['is_Lockdown']          = 0

pred_v2_12m_rounded = np.round(model_v2.predict(future_v2[features_v2])).astype(int)

print("\n🔮 V2.0 NEXT 12 MONTHS PREDICTIONS:")
for i, date in enumerate(future_12m):
    print(f"   {date.strftime('%b %Y')}: {pred_v2_12m_rounded[i]:,} arrivals ({pred_v2_12m_rounded[i]/1000:.1f}k)")

fig_v2_12m = go.Figure()
fig_v2_12m.add_trace(go.Scatter(x=future_12m, y=pred_v2_12m_rounded, mode='lines+markers',
    name='V2.0 Predictions', line=dict(color='red', width=3), marker=dict(size=6)))
fig_v2_12m.update_layout(title='V2.0 - Next 12 Months Predictions', xaxis_title='Month',
    yaxis_title='Predicted Arrivals', template='plotly_white', height=400, width=1000)
fig_v2_12m.update_xaxes(tickformat='%b %Y', tickangle=45)


🌦️  XGBoost VERSION 2.0

📊 V2 Data Split:
  Training: 2016-01-01 to 2023-12-01 (96 months)
  Testing:  2024-01-01 to 2025-12-01 (24 months)

Predicted vs Actual (first 10 test rows):
   Actual  Predicted
0   72264      77006
1   65976      92964
2   67741      78571
3   71858      94433
4   72783      59280
5   66344      40162
6   71180      41055
7   74329      30454
8   63784      18220
9   66896      32780

📈 V2.0 TEST PERFORMANCE:
   MAE:      29,360 arrivals
   RMSE:     32,376 arrivals
   MAPE:     39.11%
   Accuracy: 60.89%

🔮 V2.0 NEXT 12 MONTHS PREDICTIONS:
   Jan 2026: 28,589 arrivals (28.6k)
   Feb 2026: 31,502 arrivals (31.5k)
   Mar 2026: 43,142 arrivals (43.1k)
   Apr 2026: 52,731 arrivals (52.7k)
   May 2026: 49,483 arrivals (49.5k)
   Jun 2026: 34,930 arrivals (34.9k)
   Jul 2026: 18,948 arrivals (18.9k)
   Aug 2026: 35,895 arrivals (35.9k)
   Sep 2026: 18,948 arrivals (18.9k)
   Oct 2026: 19,837 arrivals (19.8k)
   Nov 2026: 12,607 arrivals (12.6k)
   Dec 2026: 26,38

In [126]:
# ============================================================
# 6. VERSION 3.0 — + Lag, Time & Optuna-Optimized Params
# ============================================================
print("\n" + "="*80)
print("🔬 XGBoost VERSION 3.0 (Optuna-Optimized)")
print("="*80)

df_v3 = df.copy()
df_v3['Lag_1']           = df_v3['Arrivals'].shift(1)
df_v3['Lag_2']           = df_v3['Arrivals'].shift(2)
df_v3['Rolling_Mean_3m'] = df_v3['Arrivals'].rolling(window=3, min_periods=1).mean()
df_v3['Year']            = df_v3.index.year
df_v3['Month_Num']       = df_v3.index.month
df_v3['Growth_Rate']     = df_v3['Arrivals'].pct_change() * 100
df_v3['Sudden_Increase'] = (df_v3['Growth_Rate'] > df_v3['Growth_Rate'].quantile(0.75)).astype(int)
df_v3 = df_v3.dropna()

features_v3 = [
    'Peak_Season', 'Philippine_Holidays', 'Top10Market_Holidays',
    'Avg_HighTemp', 'Avg_LowTemp', 'Precipitation',
    'Lag_1', 'Lag_2', 'Rolling_Mean_3m',
    'Year', 'Month_Num', 'Growth_Rate', 'Sudden_Increase',
    'Inflation_Rate', 'is_December', 'is_Lockdown'
]
X_v3 = df_v3[features_v3]
y_v3 = df_v3['Arrivals']

X_train_v3, X_test_v3 = X_v3.iloc[:split_point], X_v3.iloc[split_point:]
y_train_v3, y_test_v3 = y_v3.iloc[:split_point], y_v3.iloc[split_point:]

print(f"\n📊 V3 Data Split:")
print(f"  Training: {X_train_v3.index.min().date()} to {X_train_v3.index.max().date()} ({len(X_train_v3)} months)")
print(f"  Testing:  {X_test_v3.index.min().date()} to {X_test_v3.index.max().date()} ({len(X_test_v3)} months)")

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600, step=50),
        'max_depth':        trial.suggest_int('max_depth', 2, 6),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.1, 2.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.1, 2.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'gamma':            trial.suggest_float('gamma', 0, 3),
    }
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    for train_idx, val_idx in tscv.split(X_train_v3):
        model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1)
        model.fit(X_train_v3.iloc[train_idx], y_train_v3.iloc[train_idx])
        preds = model.predict(X_train_v3.iloc[val_idx])
        cv_scores.append(mean_absolute_error(y_train_v3.iloc[val_idx], preds))
    return np.mean(cv_scores)

print("\n🎯 Running Optuna optimization (100 trials, 3-fold Time Series CV)...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\nBest CV MAE:     {study.best_value:.2f}")
print(f"Best parameters: {study.best_params}")
best_params = study.best_params

model_v3 = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
model_v3.fit(X_train_v3, y_train_v3)

y_pred_train_v3_rounded = np.round(model_v3.predict(X_train_v3)).astype(int)
y_pred_test_v3_rounded  = np.round(model_v3.predict(X_test_v3)).astype(int)
train_metrics_v3 = evaluate_model(y_train_v3, y_pred_train_v3_rounded, "V3.0 TRAIN")
test_metrics_v3  = evaluate_model(y_test_v3,  y_pred_test_v3_rounded,  "V3.0 TEST")

print("\nPredicted vs Actual (first 10 test rows):")
print(pd.DataFrame({'Actual': y_test_v3.values, 'Predicted': y_pred_test_v3_rounded}).head(10))

print("\n📈 V3.0 TEST PERFORMANCE:")
print(f"   MAE:      {test_metrics_v3['mae']:,.0f} arrivals")
print(f"   RMSE:     {test_metrics_v3['rmse']:,.0f} arrivals")
print(f"   MAPE:     {test_metrics_v3['mape']:.2f}%")
print(f"   Accuracy: {test_metrics_v3['accuracy']:.2f}%")

fig_v3_test = go.Figure()
fig_v3_test.add_trace(go.Scatter(x=X_test_v3.index, y=y_test_v3, mode='lines+markers', name='Actual',
    line=dict(color='blue', width=2), marker=dict(size=8)))
fig_v3_test.add_trace(go.Scatter(x=X_test_v3.index, y=y_pred_test_v3_rounded, mode='lines+markers', name='Predicted',
    line=dict(color='red', width=2, dash='dash'), marker=dict(size=8, symbol='circle')))
fig_v3_test.update_layout(title='V3.0 - Actual vs Predicted Arrivals', xaxis_title='Month',
    yaxis_title='Arrivals', hovermode='x unified', template='plotly_white', height=450, width=1000)
fig_v3_test.update_xaxes(tickformat='%b %Y', tickangle=45)

# Future predictions V3 (static lag values from last known data)
last_values = df_v3['Arrivals'].values[-3:]
future_v3 = pd.DataFrame(index=future_12m)
future_v3['Peak_Season']          = future_v3.index.month.isin([8, 12]).astype(int)
future_v3['Philippine_Holidays']  = future_v3.index.month.map(ph_holiday_map)
future_v3['Top10Market_Holidays'] = 15
future_v3['Avg_HighTemp']         = future_v3.index.month.map(temp_high_map)
future_v3['Avg_LowTemp']          = future_v3.index.month.map(temp_low_map)
future_v3['Precipitation']        = future_v3.index.month.map(precip_map)
future_v3['Inflation_Rate']       = 3.0
future_v3['is_December']          = (future_v3.index.month == 12).astype(int)
future_v3['is_Lockdown']          = 0
future_v3['Year']                 = future_v3.index.year
future_v3['Month_Num']            = future_v3.index.month
future_v3['Lag_1']                = last_values[-1]
future_v3['Lag_2']                = last_values[-2]
future_v3['Rolling_Mean_3m']      = last_values.mean()
future_v3['Growth_Rate']          = (last_values[-1] - last_values[-2]) / last_values[-2] * 100
future_v3['Sudden_Increase']      = 0

pred_v3_12m_rounded = np.round(model_v3.predict(future_v3[features_v3])).astype(int)

print("\n🔮 V3.0 NEXT 12 MONTHS PREDICTIONS:")
for i, date in enumerate(future_12m):
    print(f"   {date.strftime('%b %Y')}: {pred_v3_12m_rounded[i]:,} arrivals ({pred_v3_12m_rounded[i]/1000:.1f}k)")

fig_v3_12m = go.Figure()
fig_v3_12m.add_trace(go.Scatter(x=future_12m, y=pred_v3_12m_rounded, mode='lines+markers',
    name='V3.0 Predictions', line=dict(color='red', width=3), marker=dict(size=6)))
fig_v3_12m.update_layout(title='V3.0 - Next 12 Months Predictions', xaxis_title='Month',
    yaxis_title='Predicted Arrivals', template='plotly_white', height=400, width=1000)
fig_v3_12m.update_xaxes(tickformat='%b %Y', tickangle=45)


🔬 XGBoost VERSION 3.0 (Optuna-Optimized)

📊 V3 Data Split:
  Training: 2016-03-01 to 2024-02-01 (96 months)
  Testing:  2024-03-01 to 2025-12-01 (22 months)

🎯 Running Optuna optimization (100 trials, 3-fold Time Series CV)...


Best trial: 96. Best value: 16331.8: 100%|██████████| 100/100 [00:49<00:00,  2.03it/s]



Best CV MAE:     16331.78
Best parameters: {'n_estimators': 550, 'max_depth': 2, 'learning_rate': 0.08113237306033798, 'subsample': 0.6012998501603662, 'colsample_bytree': 0.7334445294445897, 'reg_alpha': 0.6763091863769399, 'reg_lambda': 0.8970998766973688, 'min_child_weight': 2, 'gamma': 0.4373826199057907}

Predicted vs Actual (first 10 test rows):
   Actual  Predicted
0   67741      67544
1   71858      70716
2   72783      65919
3   66344      65071
4   71180      70363
5   74329      69469
6   63784      58762
7   66896      62132
8   79680      78645
9   84679      82792

📈 V3.0 TEST PERFORMANCE:
   MAE:      2,519 arrivals
   RMSE:     3,375 arrivals
   MAPE:     3.40%
   Accuracy: 96.60%

🔮 V3.0 NEXT 12 MONTHS PREDICTIONS:
   Jan 2026: 74,555 arrivals (74.6k)
   Feb 2026: 75,999 arrivals (76.0k)
   Mar 2026: 75,259 arrivals (75.3k)
   Apr 2026: 74,098 arrivals (74.1k)
   May 2026: 73,948 arrivals (73.9k)
   Jun 2026: 73,376 arrivals (73.4k)
   Jul 2026: 73,350 arrivals (73.3k

In [127]:
# ============================================================
# 7. FINAL COMPARISON SUMMARY
# ============================================================
print("\n" + "="*80)
print("📊 FINAL SUMMARY - ALL VERSIONS COMPARISON")
print("="*80)

print("\n📈 TEST PERFORMANCE COMPARISON:")
print("-" * 60)
print(f"{'Metric':<14} {'V1.0':>10} {'V2.0':>10} {'V3.0':>10}")
print("-" * 60)
print(f"{'MAE':<15} {test_metrics_v1['mae']:>10,.0f} {test_metrics_v2['mae']:>10,.0f} {test_metrics_v3['mae']:>10,.0f}")
print(f"{'RMSE':<15} {test_metrics_v1['rmse']:>10,.0f} {test_metrics_v2['rmse']:>10,.0f} {test_metrics_v3['rmse']:>10,.0f}")
print(f"{'MAPE':<15} {test_metrics_v1['mape']:>9,.2f}% {test_metrics_v2['mape']:>9,.2f}% {test_metrics_v3['mape']:>9,.2f}%")
print(f"{'Accuracy':<15} {test_metrics_v1['accuracy']:>9,.2f}% {test_metrics_v2['accuracy']:>9,.2f}% {test_metrics_v3['accuracy']:>9,.2f}%")

print("\n🔮 FUTURE PREDICTIONS COMPARISON (Next 12 Months):")
print("-" * 70)
print(f"{'Month':<12} {'V1.0':>10} {'V2.0':>10} {'V3.0':>10}")
print("-" * 70)
for i, date in enumerate(future_12m):
    print(f"{date.strftime('%b %Y'):<12} "
          f"{pred_v1_12m_rounded[i]/1000:>9.1f}k "
          f"{pred_v2_12m_rounded[i]/1000:>9.1f}k "
          f"{pred_v3_12m_rounded[i]/1000:>9.1f}k")

accuracies = [test_metrics_v1['accuracy'], test_metrics_v2['accuracy'], test_metrics_v3['accuracy']]
best_idx   = accuracies.index(max(accuracies))
best_model = ['V1.0 Baseline', 'V2.0 Weather+Economic', 'V3.0 Optimized'][best_idx]

print("\n" + "="*80)
print(f"🏆 BEST MODEL: {best_model} with {max(accuracies):.2f}% accuracy")
print("="*80)


📊 FINAL SUMMARY - ALL VERSIONS COMPARISON

📈 TEST PERFORMANCE COMPARISON:
------------------------------------------------------------
Metric               V1.0       V2.0       V3.0
------------------------------------------------------------
MAE                 24,720     29,360      2,519
RMSE                28,661     32,376      3,375
MAPE                32.58%     39.11%      3.40%
Accuracy            67.42%     60.89%     96.60%

🔮 FUTURE PREDICTIONS COMPARISON (Next 12 Months):
----------------------------------------------------------------------
Month              V1.0       V2.0       V3.0
----------------------------------------------------------------------
Jan 2026          43.6k      28.6k      74.6k
Feb 2026          42.0k      31.5k      76.0k
Mar 2026          43.6k      43.1k      75.3k
Apr 2026          56.4k      52.7k      74.1k
May 2026          43.6k      49.5k      73.9k
Jun 2026          43.6k      34.9k      73.4k
Jul 2026          42.0k      18.9k      73.3

In [128]:
import os
import webbrowser
import plotly.io as pio

# ============================================================
# COMBINED DASHBOARD — all charts + summary table in one tab
# ============================================================
charts = [
    ("V1.0 — Actual vs Predicted (Test Set)",  fig_v1_test),
    ("V1.0 — Next 12 Months Forecast",         fig_v1_12m),
    ("V2.0 — Actual vs Predicted (Test Set)",  fig_v2_test),
    ("V2.0 — Next 12 Months Forecast",         fig_v2_12m),
    ("V3.0 — Actual vs Predicted (Test Set)",  fig_v3_test),
    ("V3.0 — Next 12 Months Forecast",         fig_v3_12m),
]

# ── Build performance comparison table ──────────────────────
def _row(label, v1, v2, v3, fmt):
    best = min([v1, v2, v3])
    def cell(v):
        style = ' style="background:#d4edda;font-weight:bold"' if v == best else ''
        return f'<td{style}>{fmt.format(v)}</td>'
    return f'<tr><td>{label}</td>{cell(v1)}{cell(v2)}{cell(v3)}</tr>'

perf_table = f"""
<table>
  <thead>
    <tr><th>Metric</th><th>V1.0</th><th>V2.0</th><th>V3.0</th></tr>
  </thead>
  <tbody>
    {_row("MAE",      test_metrics_v1['mae'],      test_metrics_v2['mae'],      test_metrics_v3['mae'],      "{:,.0f}")}
    {_row("RMSE",     test_metrics_v1['rmse'],     test_metrics_v2['rmse'],     test_metrics_v3['rmse'],    "{:,.0f}")}
    {_row("MAPE (%)", test_metrics_v1['mape'],     test_metrics_v2['mape'],     test_metrics_v3['mape'],    "{:.2f}%")}
  </tbody>
</table>"""

# Accuracy rows (higher = better, invert best logic)
acc_best = max(test_metrics_v1['accuracy'], test_metrics_v2['accuracy'], test_metrics_v3['accuracy'])
def _acc_cell(v):
    style = ' style="background:#d4edda;font-weight:bold"' if v == acc_best else ''
    return f'<td{style}>{v:.2f}%</td>'

perf_table = perf_table.replace(
    "</tbody>",
    f'<tr><td>Accuracy (%)</td>'
    f'{_acc_cell(test_metrics_v1["accuracy"])}'
    f'{_acc_cell(test_metrics_v2["accuracy"])}'
    f'{_acc_cell(test_metrics_v3["accuracy"])}'
    f'</tr></tbody>'
)

# ── Build 12-month forecast comparison table ─────────────────
forecast_rows = ""
for i, d in enumerate(future_12m):
    forecast_rows += (
        f'<tr><td>{d.strftime("%b %Y")}</td>'
        f'<td>{pred_v1_12m_rounded[i]:,}</td>'
        f'<td>{pred_v2_12m_rounded[i]:,}</td>'
        f'<td>{pred_v3_12m_rounded[i]:,}</td></tr>\n'
    )

forecast_table = f"""
<table>
  <thead>
    <tr><th>Month</th><th>V1.0</th><th>V2.0</th><th>V3.0</th></tr>
  </thead>
  <tbody>
    {forecast_rows}
  </tbody>
</table>"""

accuracies = [test_metrics_v1['accuracy'], test_metrics_v2['accuracy'], test_metrics_v3['accuracy']]
best_version = ['V1.0 Baseline', 'V2.0 Weather+Economic', 'V3.0 Optimized'][accuracies.index(max(accuracies))]

# ── Assemble full HTML ────────────────────────────────────────
html_parts = ["""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>XGBoost Tourist Forecasting Dashboard</title>
  <style>
    body  { font-family: Arial, sans-serif; background:#f5f5f5; margin:0; padding:20px; }
    h1    { text-align:center; color:#333; }
    h2    { color:#555; margin-top:40px; border-bottom:2px solid #ddd; padding-bottom:6px; }
    .chart-box, .table-box {
      background:white; border-radius:8px; padding:20px;
      margin-bottom:30px; box-shadow:0 2px 6px rgba(0,0,0,.1);
    }
    table { border-collapse:collapse; width:100%; font-size:14px; }
    th    { background:#4a90d9; color:white; padding:10px 14px; text-align:center; }
    td    { padding:9px 14px; border-bottom:1px solid #eee; text-align:center; }
    tr:hover td { background:#f0f7ff; }
    .badge { display:inline-block; background:#28a745; color:white;
             padding:6px 14px; border-radius:20px; font-size:15px; margin-top:8px; }
  </style>
</head>
<body>
<h1>&#127965;&#65039; XGBoost Tourist Accommodation Demand Forecasting</h1>
"""]

# Charts
for title, fig in charts:
    fig_html = pio.to_html(fig, full_html=False,
                           include_plotlyjs='cdn' if title == charts[0][0] else False)
    html_parts.append(f'<div class="chart-box"><h2>{title}</h2>{fig_html}</div>\n')

# Summary tables
html_parts.append(f"""
<div class="table-box">
  <h2>&#128202; Final Summary — Test Performance Comparison</h2>
  <p style="font-size:13px;color:#888;">Green highlight = best value per metric.</p>
  {perf_table}
  <p class="badge">&#127942; Best Model: {best_version} &nbsp;({max(accuracies):.2f}% accuracy)</p>
</div>

<div class="table-box">
  <h2>&#128302; Future Predictions Comparison (Next 12 Months)</h2>
  {forecast_table}
</div>
</body></html>
""")

output_path = os.path.join(os.path.dirname(CSV_PATH), "xgboost_dashboard.html")
with open(output_path, "w", encoding="utf-8") as f:
    f.write("".join(html_parts))

print(f"✅ Dashboard saved to: {output_path}")
webbrowser.open(f"file:///{output_path.replace(os.sep, '/')}")

✅ Dashboard saved to: c:\Users\User\Pictures\For XGBoost\xgboost_dashboard.html


True